# Option Pricing Model (OPM) – Project Horizon Structured M&A Option

Values the Call/Put option structure between Olympus BV (call holder) and Titan (put holder).

## Key Contract Terms (from Option Agreement)
- **Option Shares:** 1,386,020 E ordinary shares
- **Call Option Period:** exercisable Year 3 – Year 5 from SPA Completion (Bermudan)
- **Put Option Period:** exercisable Year 3.5 – Year 6 from SPA Completion (Bermudan)
- **Strike = SPA price + Applicable Coupon** (step-up schedule, capitalised annually)
  - Call coupons: 2.75% / 3.00% / 3.00% / 4.20% / 4.20% (Years 0-1 through 4-5)
  - Put coupons:  2.75% / 3.00% / 3.00% / 4.10% / 4.10% / 4.10% (Years 0-1 through 5-6)
- **Pricing method:** Least-Squares Monte Carlo (Longstaff-Schwartz) for Bermudan exercise

## Simplifying Assumptions
- SPA price per share = 100 (placeholder; scale results to actual price)
- No Gaia leakage, no SPA claim adjustments, no dividends
- Risk-neutral drift = risk-free rate (standard option pricing)

## 1. Inputs

In [ ]:
import numpy as np
from numpy.polynomial import polynomial as P

# ============================================================
# CONTRACT TERMS (from Option Agreement)
# ============================================================
# Total equity = $3.2bn; option shares = 25% of total equity
TOTAL_EQUITY_USD = 3_200_000_000
OPTION_SHARES    = 1_386_020       # E ordinary shares (25% stake)
TOTAL_SHARES     = OPTION_SHARES / 0.25
SPA_price        = TOTAL_EQUITY_USD / TOTAL_SHARES   # USD per share = 577.27

# Step-up coupon schedules – capitalised annually (from Schedule 3 definitions)
CALL_COUPONS = [0.0275, 0.0300, 0.0300, 0.0420, 0.0420]          # Years 0-1 … 4-5
PUT_COUPONS  = [0.0275, 0.0300, 0.0300, 0.0410, 0.0410, 0.0410]  # Years 0-1 … 5-6

# Exercise windows
call_start, call_end = 3.0, 5.0
put_start,  put_end  = 3.5, 6.0

# ============================================================
# MARKET / ECONOMIC ASSUMPTIONS
# ============================================================
# Asset volatility = unlevered (historical) equity vol from global KO bottler peers.
# Peer-implied asset vol = 21.6%; rounded to nearest 5% = 20%.
# In OPM the σ parameter is forward expected asset vol; historical unlevered equity vol is the standard proxy.
asset_volatility = 0.20

risk_free_rate   = 0.04929  # USD risk-free rate at valuation date (per WACC)

# ============================================================
# SIMULATION SETTINGS
# ============================================================
simulations    = 100_000
steps_per_year = 12
dt             = 1 / steps_per_year
total_years    = 6
total_steps    = int(total_years / dt) + 1  # +1 to include endpoint (t=6)

np.random.seed(42)
print(f"SPA price per share:        USD {SPA_price:,.2f}")
print(f"Option shares value at SPA: USD {SPA_price * OPTION_SHARES:,.0f}  (25% of ${TOTAL_EQUITY_USD/1e9:.1f}bn)")
print(f"Asset volatility:           {asset_volatility:.0%}")
print(f"Risk-free rate:             {risk_free_rate:.3%}")
print("Inputs loaded.")

## 2. Build Strike Paths

Strike at time *t* = SPA price × ∏(1 + rᵢ) for completed years × (1 + r_current)^fractional_year.  
Separate schedules for call and put as per the Option Agreement.

In [ ]:
def build_strike_path(SPA_price, coupons, total_steps, dt):
    """Annual compounding strike path using contract coupon schedule."""
    strike = np.zeros(total_steps)
    for step in range(total_steps):
        t        = step * dt
        yr_int   = int(t)
        frac     = t - yr_int
        acc      = SPA_price
        # Compound through completed years
        for y in range(min(yr_int, len(coupons))):
            acc *= (1 + coupons[y])
        # Partial-year accrual in current bucket (annually compounded)
        if yr_int < len(coupons):
            acc *= (1 + coupons[yr_int]) ** frac
        strike[step] = acc
    return strike

call_strike = build_strike_path(SPA_price, CALL_COUPONS, total_steps, dt)
put_strike  = build_strike_path(SPA_price, PUT_COUPONS,  total_steps, dt)

# Spot-check a few values
time_grid = np.arange(total_steps) * dt
for yr in [0, 1, 2, 3, 4, 5, 6]:
    idx = int(yr / dt)
    if idx < total_steps:
        print(f"Year {yr:3.1f} | Call strike: {call_strike[idx]:.4f} | Put strike: {put_strike[idx]:.4f}")

## 3. Simulate Equity Value Paths (Risk-Neutral GBM)

Under the risk-neutral measure the equity drifts at the risk-free rate.

In [ ]:
paths = np.zeros((simulations, total_steps))
paths[:, 0] = SPA_price

for t in range(1, total_steps):
    z = np.random.standard_normal(simulations)
    paths[:, t] = paths[:, t-1] * np.exp(
        (risk_free_rate - 0.5 * asset_volatility**2) * dt
        + asset_volatility * np.sqrt(dt) * z
    )

print(f"Simulated {simulations:,} paths × {total_steps} steps.")
print(f"Mean equity at Year 6: {paths[:, -1].mean():.2f}  (expect ~{SPA_price * np.exp(risk_free_rate * total_years):.2f} risk-neutral)")

## 4. Call Option – Bermudan LSM (Year 3–5)

Olympus BV holds the call: payoff = max(S − K_call, 0).  
We use Longstaff-Schwartz (LSM) regression to determine early exercise.

In [ ]:
def lsm_bermudan_call(paths, strike_path, window_start, window_end, dt, risk_free_rate):
    """
    Longstaff-Schwartz LSM for a Bermudan call.
    Exercise at any monthly step in [window_start, window_end].
    Returns per-share value.
    """
    idx_start = int(window_start / dt)
    idx_end   = int(window_end   / dt)
    n_sims    = paths.shape[0]
    discount  = np.exp(-risk_free_rate * dt)

    # Initialise: hold value at final exercise date
    payoff_terminal = np.maximum(paths[:, idx_end] - strike_path[idx_end], 0)
    cash_flow = payoff_terminal.copy()

    # Roll back through exercise window
    for step in range(idx_end - 1, idx_start - 1, -1):
        t = step * dt
        cash_flow *= discount  # discount one step forward

        intrinsic = paths[:, step] - strike_path[step]
        itm = intrinsic > 0

        if itm.sum() < 5:
            continue

        S_itm  = paths[itm, step]
        cf_itm = cash_flow[itm]

        # Regression: continuation value ~ polynomial in S
        X = np.column_stack([np.ones(itm.sum()), S_itm, S_itm**2, S_itm**3])
        coeffs, _, _, _ = np.linalg.lstsq(X, cf_itm, rcond=None)
        continuation = X @ coeffs

        # Exercise where intrinsic > estimated continuation
        exercise = intrinsic[itm] > continuation
        idx_itm  = np.where(itm)[0]
        cash_flow[idx_itm[exercise]] = intrinsic[itm][exercise]

    # Discount back from window_start to time 0
    value = np.exp(-risk_free_rate * window_start) * np.mean(cash_flow)
    return value

call_value_per_share = lsm_bermudan_call(
    paths, call_strike, call_start, call_end, dt, risk_free_rate
)
print(f"Call value (per share):       {call_value_per_share:>8.4f}")

## 5. Put Option – Bermudan LSM (Year 3.5–6)

Titan holds the put: payoff = max(K_put − S, 0).  
From Olympus BV's perspective this is a **liability** (short put).

In [ ]:
def lsm_bermudan_put(paths, strike_path, window_start, window_end, dt, risk_free_rate):
    """
    Longstaff-Schwartz LSM for a Bermudan put.
    Exercise at any monthly step in [window_start, window_end].
    Returns per-share value.
    """
    idx_start = int(window_start / dt)
    idx_end   = int(window_end   / dt)
    discount  = np.exp(-risk_free_rate * dt)

    payoff_terminal = np.maximum(strike_path[idx_end] - paths[:, idx_end], 0)
    cash_flow = payoff_terminal.copy()

    for step in range(idx_end - 1, idx_start - 1, -1):
        cash_flow *= discount

        intrinsic = strike_path[step] - paths[:, step]
        itm = intrinsic > 0

        if itm.sum() < 5:
            continue

        S_itm  = paths[itm, step]
        cf_itm = cash_flow[itm]

        X = np.column_stack([np.ones(itm.sum()), S_itm, S_itm**2, S_itm**3])
        coeffs, _, _, _ = np.linalg.lstsq(X, cf_itm, rcond=None)
        continuation = X @ coeffs

        exercise = intrinsic[itm] > continuation
        idx_itm  = np.where(itm)[0]
        cash_flow[idx_itm[exercise]] = intrinsic[itm][exercise]

    value = np.exp(-risk_free_rate * window_start) * np.mean(cash_flow)
    return value

put_value_per_share = lsm_bermudan_put(
    paths, put_strike, put_start, put_end, dt, risk_free_rate
)
print(f"Put value (per share):        {put_value_per_share:>8.4f}")

## 6. Summary – Net Instrument Value

From **Olympus BV's** perspective:  
- Long Call (asset) + Short Put (liability)  
- Net = Call − Put

In [ ]:
net_per_share = call_value_per_share - put_value_per_share

call_total = call_value_per_share * OPTION_SHARES
put_total  = put_value_per_share  * OPTION_SHARES
net_total  = net_per_share        * OPTION_SHARES

print("=" * 55)
print(f"  Assumptions: asset_vol={asset_volatility:.0%}, rf={risk_free_rate:.3%}, SPA=USD {SPA_price:,.2f}")
print("-" * 55)
print(f"  Call value   per share: USD {call_value_per_share:>10.4f}")
print(f"  Put value    per share: USD {put_value_per_share:>10.4f}")
print(f"  Net value    per share: USD {net_per_share:>10.4f}")
print("-" * 55)
print(f"  Call value   total:     USD {call_total:>14,.0f}")
print(f"  Put value    total:     USD {put_total:>14,.0f}")
print(f"  Net value    total:     USD {net_total:>14,.0f}")
print("=" * 55)

## 7. Sensitivity Analysis

Net per-share value across volatility and risk-free rate assumptions.

> This re-runs the full LSM for each scenario — takes ~1–2 minutes.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.set_xlim(0, 7)
ax.set_ylim(-0.5, 2.5)
ax.set_xlabel("Years from SPA Completion", fontsize=11)
ax.set_yticks([0.5, 1.5])
ax.set_yticklabels(["Titan\n(Put holder)", "Olympus BV\n(Call holder)"], fontsize=10)
ax.set_xticks(range(8))
ax.grid(axis='x', linestyle='--', alpha=0.4)
ax.set_facecolor('#f9f9f9')

# ── Call bar (Olympus, top row y=1) ─────────────────────────
ax.broken_barh([(3, 2)], (1.05, 0.9), facecolor='#2196F3', alpha=0.85, label='Call window (Olympus BV)')

# ── Put bar (Titan, bottom row y=0) ──────────────────────────
ax.broken_barh([(3.5, 2.5)], (0.05, 0.9), facecolor='#FF7043', alpha=0.85, label='Put window (Titan)')

# ── Overlap shading ──────────────────────────────────────────
ax.broken_barh([(3.5, 1.5)], (0.05, 1.9), facecolor='#9C27B0', alpha=0.18)
ax.axvline(3.5, color='#9C27B0', linestyle=':', linewidth=1.5)
ax.axvline(5.0, color='#9C27B0', linestyle=':', linewidth=1.5)
ax.text(4.25, 2.25, 'Overlap\n(Yr 3.5–5)', ha='center', va='top', fontsize=8.5,
        color='#6A1B9A', fontweight='bold')

# ── Annotations ──────────────────────────────────────────────
ax.annotate('Call opens\n(Yr 3)', xy=(3, 1.5), xytext=(2.4, 2.1),
            arrowprops=dict(arrowstyle='->', color='#1565C0'), fontsize=8, color='#1565C0')
ax.annotate('Call expires\n(Yr 5)', xy=(5, 1.5), xytext=(5.2, 2.1),
            arrowprops=dict(arrowstyle='->', color='#1565C0'), fontsize=8, color='#1565C0')
ax.annotate('Put opens\n(Yr 3.5)', xy=(3.5, 0.5), xytext=(2.4, -0.3),
            arrowprops=dict(arrowstyle='->', color='#BF360C'), fontsize=8, color='#BF360C')
ax.annotate('Put expires\n(Yr 6)', xy=(6, 0.5), xytext=(6.1, -0.3),
            arrowprops=dict(arrowstyle='->', color='#BF360C'), fontsize=8, color='#BF360C')

# ── "Olympus only" and "Titan only" labels ───────────────────
ax.text(3.25, 0.5, 'Call only\n(0.5yr)', ha='center', va='center', fontsize=7.5,
        color='white', fontweight='bold')
ax.text(5.5, 0.5, 'Put only\n(1yr)', ha='center', va='center', fontsize=7.5,
        color='white', fontweight='bold')

legend = ax.legend(loc='upper left', fontsize=9, framealpha=0.9)
ax.set_title("Project Horizon – Option Exercise Windows", fontsize=12, fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig("option_windows_gantt.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: option_windows_gantt.png")

In [ ]:
vol_range = [0.15, 0.20, 0.25, 0.30, 0.35]   # asset vol range
rf_range  = [0.04, 0.04929, 0.06]

SIM_SENS  = 50_000
np.random.seed(42)

print(f"Net value to Olympus BV (USD millions) | base case marked with *")
print(f"{'AssetVol':>9} | " + " | ".join(f"  rf={r:.2%} " for r in rf_range))
print("-" * (13 + 15 * len(rf_range)))

for vol in vol_range:
    row = []
    for rf in rf_range:
        p = np.zeros((SIM_SENS, total_steps))
        p[:, 0] = SPA_price
        for t in range(1, total_steps):
            z = np.random.standard_normal(SIM_SENS)
            p[:, t] = p[:, t-1] * np.exp(
                (rf - 0.5 * vol**2) * dt + vol * np.sqrt(dt) * z
            )
        c  = lsm_bermudan_call(p, call_strike, call_start, call_end, dt, rf)
        pu = lsm_bermudan_put( p, put_strike,  put_start,  put_end,  dt, rf)
        net_m = (c - pu) * OPTION_SHARES / 1e6
        marker = " *" if (vol == 0.20 and rf == 0.04929) else "  "
        row.append(f"{net_m:>8.1f}{marker}")
    print(f"{vol:>8.0%}   | " + " | ".join(row))